In [1]:
try:
    spark.stop()
except Exception as e:
    pass

# ⚙️ 1. Setup Spark local (avec support Delta)


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("RecipeSearchPoC")
    # Support Delta Lake
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # Broadcast join automatique pour df_index (table d'index souvent < 200 Mo)
    .config("spark.sql.autoBroadcastJoinThreshold", "200m")
    # Gestion de la mémoire occupée par Spark
    .config("spark.executor.memory", "2g")  # Mémoire allouée à chaque nœud de calcul
    .config("spark.driver.memory", "2g")    # Mémoire allouée au chef d'orchestre
    .getOrCreate()
)

print("✅ Spark ready")


✅ Spark ready


# 📥 2. Chargement des données


In [3]:
DATA_PATH = "../data/recipes_parquets"

# Delta gère nativement le Data Skipping et le Z-Order définis dans le pipeline
df_main      = spark.read.format("delta").load(f"{DATA_PATH}/recipes_main")
df_index     = spark.read.format("delta").load(f"{DATA_PATH}/ingredients_index")
df_nutrition = spark.read.format("delta").load(f"{DATA_PATH}/recipes_nutrition_detail")

print("✅ Data loaded (lazy — aucun scan déclenché)")


✅ Data loaded (lazy — aucun scan déclenché)


# 🔗 3. Vue master


In [4]:
# On définit le join une seule fois — Spark ne l'exécute qu'à la demande
df_master = df_main.join(df_nutrition, on="recipe_id", how="left")

print(f"Colonnes master ({len(df_master.columns)}) : {df_master.columns}")


Colonnes master (22) : ['recipe_id', 'title', 'description', 'instructions_text', 'ingredients_raw', 'ingredients_validated', 'n_ingredients_validated', 'n_steps', 'cook_minutes', 'cook_time_category', 'image_url', 'image_urls', 'has_image', 'source_url', 'energy_kcal', 'nutri_score', 'tags', 'fat_g', 'protein_g', 'salt_g', 'saturates_g', 'sugars_g']


# 🔎 4. Recherche par nom


In [5]:
def search_by_name(query: str, limit: int = 20) -> list[dict]:
    rows = (
        df_master
        .filter(col("title").ilike(f"%{query}%"))
        .select("recipe_id", "title", "cook_minutes", "nutri_score", "image_url")
        .limit(limit)
        .collect()
    )
    return [r.asDict() for r in rows]

# Test
search_by_name("pasta")


26/04/08 17:55:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[{'recipe_id': 'c7372c374e',
  'title': 'Homemade Fettuccini Pasta Dough',
  'cook_minutes': None,
  'nutri_score': 'E',
  'image_url': 'https://s-media-cache-ak0.pinimg.com/736x/92/40/ff/9240ffe585bacb3addf8a3a26ba5ccdc.jpg'},
 {'recipe_id': 'e53039454b',
  'title': 'Egyptian Koshary Pasta',
  'cook_minutes': None,
  'nutri_score': 'E',
  'image_url': 'https://thumbs.dreamstime.com/z/egyptian-food-koshary-pasta-sauce-traditional-made-spaghetti-chickpeas-rice-onion-74514441.jpg'},
 {'recipe_id': '7b22c46b8e',
  'title': 'Greek Orzo Pasta Salad',
  'cook_minutes': 20,
  'nutri_score': 'E',
  'image_url': 'http://www.hummusapien.com/wp-content/uploads/2013/09/greeksalad21.jpg'},
 {'recipe_id': '309ca5c971',
  'title': 'Cucumber Pasta',
  'cook_minutes': None,
  'nutri_score': 'E',
  'image_url': 'http://cdn.skim.gs/image/upload/c_fill,dpr_1.0,f_auto,fl_lossy,q_auto,w_940/v1456344637/msi/cucumber_noodles_3_ayjyye.jpg'},
 {'recipe_id': '27085e38d7',
  'title': 'Crabstick pasta salad',
  'c

# 🥕 5. Recherche par ingrédient (le cœur)


In [6]:
def search_by_ingredient(ingredient):
    """
    df_index est typiquement < 200 Mo → AQE le broadcast automatiquement.
    Le join devient un map-side join, pas de shuffle sur df_master.
    """
    recipe_ids = (
        df_index
        .filter(col("ingredient").ilike(f"%{ingredient}%"))
        .select("recipe_id")
        .distinct()
    )
    return (
        df_master
        .join(recipe_ids, on="recipe_id")
        .select("recipe_id", "title", "cook_minutes", "nutri_score", "image_url", "ingredients_raw")
        .limit(20)
    )

# Test
search_by_ingredient("tomato").toPandas()


ImportError: Pandas >= 1.0.5 must be installed; however, it was not found.

# 🧠 6. Recherche avancée (clé du PoC)


In [ ]:
def search_advanced(name=None, ingredient=None, max_time=None, nutri_score=None):
    """
    Les filtres sont poussés avant le join (predicate pushdown).
    Spark n'ouvre que les fichiers Delta concernés.
    """
    df = df_master

    if name:
        df = df.filter(col("title").ilike(f"%{name}%"))

    if max_time:
        df = df.filter(col("cook_minutes") <= max_time)

    if nutri_score:
        df = df.filter(col("nutri_score") == nutri_score)

    # Le join sur df_index vient EN DERNIER pour profiter des filtres précédents
    if ingredient:
        recipe_ids = (
            df_index
            .filter(col("ingredient").ilike(f"%{ingredient}%"))
            .select("recipe_id")
            .distinct()
        )
        df = df.join(recipe_ids, on="recipe_id")

    return df.select(
        "recipe_id",
        "title",
        "cook_minutes",
        "cook_time_category",
        "nutri_score",
        "energy_kcal",
        "image_url",
        "ingredients_raw",
    ).limit(20)

# Test : recettes rapides avec nutri_score A
search_advanced(max_time=30, nutri_score="A").toPandas()


# 🍽️ 7. Affichage recette complète


In [ ]:
def show_recipe(recipe_id):
    """
    .limit(1).collect() = Spark s'arrête dès le 1er match,
    pas de scan complet de la table.
    """
    recipe = (
        df_master
        .filter(col("recipe_id") == recipe_id)
        .limit(1)
        .collect()
    )

    if not recipe:
        print("❌ Recette introuvable")
        return

    r = recipe[0].asDict()

    print("🍽️ ", r.get("title", "—"))
    print("⏱️  Temps      :", r.get("cook_minutes"), "min  [", r.get("cook_time_category"), "]")
    print("🔥 Calories   :", r.get("energy_kcal"), "kcal")
    print("🥗 Nutri-score:", r.get("nutri_score"))
    print("🖼️  Image      :", r.get("image_url"))
    print("\n📝 Ingrédients:")
    print(r.get("ingredients_raw", "—"))
    print("\n👨‍🍳 Instructions:")
    print(r.get("instructions_text", "—"))

# Test : 1er recipe_id disponible
first_id = df_master.select("recipe_id").first()["recipe_id"]
show_recipe(first_id)


# 🛑 8. Arrêt Spark


In [ ]:
spark.stop()
print("✅ Spark stopped")
